# Q12 — Self-calibration and transfer under uncertain timing

**Upload this notebook to Jupyter on your server and select Run All. Only this file is needed.**
The notebook embeds the experiment code and reuses your existing P1 NetSquid/Qoala environment.
It does not install packages or change the Qoala checkout. Keep the server allocation alive while it runs.

The default `MODE="full"` runs **pilot → own-job calibration → frozen victim evaluation → export** automatically.
`WORKERS=6` leaves two of your eight cores available. If all eight allocated cores are free, you can use `WORKERS=8`.
Each worker runs a separate simulator process with numerical-library threads limited to one.

**Question:** Can a phase learned from the attacker's own quantum test applications transfer to another tenant
when the request time is only approximately known? Selection uses only the owner's returned verification bits.
The execution policy receives a declared coarse request epoch and the previously frozen phase, without victim outcomes,
actual start times, or queue traces. Traces are retained separately for evaluator validation.

**Scope:** Shared placement, access to own calibration applications, workload-class knowledge, and a supplied coarse
request epoch are assumptions. This notebook does not demonstrate a deployed API for obtaining that epoch.
Calibration and transfer use the same workload class and simulator parameters, with fresh inputs and random streams.
Both workloads use FCFS, ideal local gates and entanglement generation, and memory dephasing with T2 = 4 ms.
Each execution resets the simulator; this is not a sustained-load experiment.

| Stage | Executions | Purpose |
|---|---:|---|
| Integration pilot | 1,296 | Noise-free correctness, correction branches, timing and equal-service checks |
| Own-job calibration | 294,912 | 32 independent campaigns per workload; 16 candidate phases and two controls |
| Victim evaluation | 98,304 | Learned schedule, equal-work random schedule, and no competitor |

The two primary contrasts use **±0.5 ms request-time uncertainty**. Four secondary conditions test ±1 ms,
±2 ms, benign background traffic, and a changed message-delay distribution. Every competing schedule submits
the same four 30-instruction CPU jobs, totaling 1.2 ms of CPU service per execution.

**Validation status:** The design, analysis and checkpoint logic were checked locally. The new Q12 adapter has
not been executed with NetSquid here. The mandatory server pilot must pass before calibration starts.
Passing the pilot or completing the study does not establish a positive experimental result.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.home() / "P1"
MODE = "full"
WORKERS = 6

# "auto": resume the latest run created by this exact code bundle, or start one.
# None: explicitly start a NEW run. A full path: resume that particular run.
RESUME_RUN_DIR = "auto"

# Usually leave these as None. The next cells locate your existing environment.
PYTHON = None   # Example: PROJECT_DIR / ".venv-p1" / "bin" / "python"
REPO = None     # Example: PROJECT_DIR / "qoala-sim"

# The review ZIP includes outcomes, analysis, code, metadata and diagnostic logs.
# Raw traces stay on the server. Set True later if a full trace archive is needed.
EXPORT_RAW = False


In [ ]:
import base64, fcntl, gzip, hashlib, json, os, sys

SOURCE_BUNDLE_SHA256 = '470078ab4b2c014d42b97dbba496934a0195ae13a58d3404274c06b8e5b3b7b4'
_EMBEDDED = '...'
_EXPECTED_FILES = ['q12_design.py', 'q12_engine.py', 'q12.py', 'q12_tests.py', 'p1_engine.py', 'q6_reference.py', 'time_checks.py', 'parent_sources.json']

def _sha_bytes(value):
    return hashlib.sha256(value).hexdigest()

def install_embedded_code(project):
    data = gzip.decompress(base64.b64decode(_EMBEDDED))
    if _sha_bytes(data) != SOURCE_BUNDLE_SHA256:
        raise RuntimeError("Embedded source bundle checksum mismatch")
    files = json.loads(data)
    if set(files) != set(_EXPECTED_FILES):
        raise RuntimeError("Unexpected embedded source files")
    base = Path(project).expanduser().resolve() / "experiment_code"
    base.mkdir(parents=True, exist_ok=True)
    folder = base / ("q12-" + SOURCE_BUNDLE_SHA256[:16])
    with (base / (folder.name + ".install.lock")).open("a+") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        folder.mkdir(exist_ok=True)
        for name, content in files.items():
            if Path(name).name != name:
                raise RuntimeError("Invalid embedded filename")
            expected = content.encode()
            destination = folder / name
            if destination.exists():
                if destination.read_bytes() != expected:
                    raise RuntimeError("Installed source differs: " + str(destination) +
                                       ". Preserve it; do not overwrite or mix versions.")
            else:
                temporary = destination.with_name(name + ".tmp-" + str(os.getpid()))
                temporary.write_bytes(expected)
                os.replace(temporary, destination)
            if _sha_bytes(destination.read_bytes()) != _sha_bytes(expected):
                raise RuntimeError("Source verification failed: " + name)
    return folder

PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
CODE_DIR = install_embedded_code(PROJECT_DIR)
print("Embedded code installed and verified:", CODE_DIR, flush=True)
print("Source bundle hash:", SOURCE_BUNDLE_SHA256, flush=True)


In [ ]:
import shutil, subprocess

if MODE not in ("pilot", "full"):
    raise ValueError('MODE must be "pilot" or "full"')
if not isinstance(WORKERS, int) or isinstance(WORKERS, bool) or WORKERS < 1:
    raise ValueError("WORKERS must be a positive integer")
available_cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if available_cpus and WORKERS > available_cpus:
    raise ValueError(f"WORKERS={WORKERS}, but this process can use only {available_cpus} CPUs")

RUN_ENV = os.environ.copy()
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    RUN_ENV[variable] = "1"
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV.pop("PYTHONOPTIMIZE", None)

repo_candidates = ([Path(REPO).expanduser()] if REPO is not None else
                   [PROJECT_DIR / name for name in ("qoala-sim", "p1-qoala", "qoala")] +
                   [Path.home() / "p1-qoala", Path.home() / "qoala-sim"])
repos = []
for candidate in repo_candidates:
    if not candidate.is_dir():
        continue
    result = subprocess.run(["git", "-C", str(candidate), "rev-parse", "HEAD"],
                            capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip() == "31fa2c9699578d227736557f35cd01a134c4bbde":
        repos.append(candidate.resolve())
if not repos:
    raise RuntimeError("Could not locate the existing pinned Qoala checkout. Set REPO in the configuration "
                       "cell to its directory, then rerun. Checked: " + ", ".join(map(str, repo_candidates)))
REPO = repos[0]

if PYTHON is not None:
    located = shutil.which(str(PYTHON))
    python_candidates = [located or str(Path(PYTHON).expanduser().absolute())]
else:
    # Do not resolve interpreter symlinks: doing so can bypass a virtual environment.
    python_candidates = [str(PROJECT_DIR / ".venv-p1/bin/python"),
                         str(PROJECT_DIR / ".venv/bin/python"), sys.executable]
python_candidates = list(dict.fromkeys(python_candidates))
preflight = "import json,sys; import q6_reference; info=q6_reference.check_repo(sys.argv[1]); " \
            "import scipy; info['scipy']=scipy.__version__; print(json.dumps(info,indent=2))"
failures = []
chosen = None
for candidate in python_candidates:
    if not Path(candidate).is_file():
        continue
    result = subprocess.run([candidate, "-c", preflight, str(REPO)], cwd=CODE_DIR,
                            env=RUN_ENV, capture_output=True, text=True)
    if result.returncode == 0:
        chosen = candidate
        print("Using Python:", candidate, flush=True)
        print("Using Qoala:", REPO, flush=True)
        print(result.stdout, flush=True)
        break
    failures.append(candidate + "\n" + result.stdout + result.stderr)
if chosen is None:
    print("\n".join(failures), flush=True)
    raise RuntimeError("No validated environment found. Set PYTHON to the same interpreter used for "
                       "Q11/Q9S. Required: Python 3.10, NetSquid 1.1.8, pydynaa 1.0.2, Qoala 2.1.0, "
                       "NumPy 1.26.4, netqasm 2.3.0 and SciPy. This notebook has not changed your environment.")
PYTHON = chosen

# Capture and print so output appears reliably in Jupyter as well as a terminal.
tests = subprocess.run([PYTHON, "-u", str(CODE_DIR / "q12_tests.py")],
                       cwd=CODE_DIR, env=RUN_ENV, capture_output=True, text=True)
print(tests.stdout, end="", flush=True)
if tests.stderr:
    print(tests.stderr, end="", flush=True)
if tests.returncode:
    raise RuntimeError("Local design checks failed; do not start the experiment")
print(f"Preflight passed. Mode: {MODE} | workers: {WORKERS}. The next cell runs the NetSquid pilot.", flush=True)


## Run or resume

The next cell prints the **RUN DIRECTORY immediately** and then streams progress. It remains busy while simulations run.
The pilot gives a rough runtime estimate; later stages report progress and updated estimates.

If the server session ends, reopen **this same notebook** and Run All again. `RESUME_RUN_DIR="auto"`
locates its latest run on disk. You can also paste the exact printed path into that variable.
Completed batches are checked and reused; an interrupted batch of up to 128 executions is rerun.
`MODE="full"` continues through calibration and evaluation without changing settings between stages.
Do not launch two coordinators for the same run. If an older session still has the process running, the run lock will reject a second launch.


In [ ]:
from datetime import datetime, timezone
import signal, time

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
POINTER = RESULTS_DIR / (".q12-latest-" + SOURCE_BUNDLE_SHA256[:16] + ".json")

def choose_run_directory(resume):
    if resume == "auto" and POINTER.exists():
        previous = json.loads(POINTER.read_text())
        if previous["source_bundle_sha256"] != SOURCE_BUNDLE_SHA256:
            raise RuntimeError("Resume pointer belongs to a different source bundle")
        root = Path(previous["run_dir"]).expanduser().resolve()
        if not root.is_dir():
            raise RuntimeError("The saved run folder is missing: " + str(root) +
                               ". Restore it, or set RESUME_RUN_DIR=None to explicitly start anew.")
        return root
    if resume is None or resume == "auto":
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        return RESULTS_DIR / ("q12-" + stamp)
    root = Path(resume).expanduser().resolve()
    if not root.is_dir():
        raise RuntimeError("Resume folder does not exist: " + str(root))
    if not root.name.startswith("q12-"):
        raise RuntimeError("Expected a Q12 run folder; do not reuse a Q11/Q9S folder")
    return root

RUN_DIR = choose_run_directory(RESUME_RUN_DIR)
if RUN_DIR == REPO or REPO in RUN_DIR.parents or RUN_DIR == CODE_DIR or CODE_DIR in RUN_DIR.parents:
    raise RuntimeError("Results must be outside the source and Qoala checkout folders")
RUN_DIR.mkdir(parents=True, exist_ok=True)
pointer_temp = POINTER.with_name(POINTER.name + ".tmp-" + str(os.getpid()))
pointer_temp.write_text(json.dumps(dict(run_dir=str(RUN_DIR), source_bundle_sha256=SOURCE_BUNDLE_SHA256), indent=2))
os.replace(pointer_temp, POINTER)

print("RUN DIRECTORY:", RUN_DIR, flush=True)
print("For an explicit resume, set RESUME_RUN_DIR =", repr(str(RUN_DIR)), flush=True)
print('Keep MODE="full" to continue all stages. The default "auto" remembers this path.', flush=True)
launcher_log = RUN_DIR / ("launcher-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".log")
command = [PYTHON, "-u", str(CODE_DIR / "q12.py"), "run", "--repo", str(REPO),
           "--run-dir", str(RUN_DIR), "--mode", MODE, "--workers", str(WORKERS)]

with launcher_log.open("w") as handle:
    process = subprocess.Popen(command, cwd=CODE_DIR, env=RUN_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    print("Coordinator PID:", process.pid, "| log:", launcher_log, flush=True)
    try:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
        exit_code = process.wait()
    except BaseException:
        # Stop the coordinator and its worker subprocesses together on interruption.
        for sig in (signal.SIGINT, signal.SIGTERM, signal.SIGKILL):
            if process.poll() is not None:
                break
            try:
                os.killpg(process.pid, sig)
            except ProcessLookupError:
                break
            try:
                process.wait(timeout=10)
                break
            except subprocess.TimeoutExpired:
                pass
        print("Stopped. Completed checkpoints remain reusable in", RUN_DIR, flush=True)
        raise
    finally:
        process.stdout.close()
if exit_code:
    raise RuntimeError("Q12 stopped with exit code " + str(exit_code) +
                       ". Run the export cell below manually and share the review ZIP, including logs.")
print("Requested stages finished. The next cell displays available results and exports them.", flush=True)


## Results and export

Run this cell after completion, or **manually after an error**. It prints status even if no analysis exists yet.
By default it creates a review ZIP with all compact outcomes, code, manifests, analysis and logs.
Large raw traces remain untouched in the run folder. A complete raw-trace audit requires those files as well;
set `EXPORT_RAW=True` and rerun only this cell if a full archive is requested.

The primary endpoint is the fraction of **wrong verification outputs per offered victim execution**.
Unfinished executions are reported separately. A positive difference means more wrong outputs under the learned schedule
than under the equal-work random schedule. This is an application outcome, not a direct general-purpose fidelity measurement.
The independent statistical units are the 32 calibration campaigns. Primary mean intervals use 20,000 campaign bootstrap
resamples with a two-comparison Bonferroni adjustment (approximate coverage). Exact sign tests address campaign direction,
a different estimand. The eight secondary comparisons are exploratory and retain pointwise intervals.


In [ ]:
import html, zipfile
from IPython.display import display, HTML, FileLink

if "RUN_DIR" not in globals():
    saved_pointer = PROJECT_DIR / "results" / (".q12-latest-" + SOURCE_BUNDLE_SHA256[:16] + ".json")
    if RESUME_RUN_DIR not in (None, "auto"):
        RUN_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    elif saved_pointer.exists():
        RUN_DIR = Path(json.loads(saved_pointer.read_text())["run_dir"])
    else:
        raise RuntimeError("No Q12 run folder is available yet; run the preceding cells first")

print("RUN DIRECTORY:", RUN_DIR, flush=True)
for label, marker in [("Pilot complete", "PILOT_COMPLETE.json"),
                      ("Calibration complete", "CALIBRATION_COMPLETE.json"),
                      ("Entire study execution complete", "COMPLETE.json")]:
    print(label + ":", (RUN_DIR / marker).exists(), flush=True)

if (RUN_DIR / "analysis.json").exists():
    report = json.loads((RUN_DIR / "analysis.json").read_text())
    headings = ["Workload", "Learned − random (percentage points)",
                "Adjusted bootstrap interval", "Adjusted direction p", "All targets finished", "Criterion met"]
    table_rows = []
    for row in report["primary_contrasts"]:
        lo, hi = row["approximate_bootstrap95_interval_pp"]
        table_rows.append([row["workload"], f'{row["mean_difference_pp"]:.3f}',
                           f'[{lo:.3f}, {hi:.3f}]', f'{row["adjusted_direction_p"]:.4g}',
                           row["all_compared_targets_completed"], row["supports_additional_wrong_outputs"]])
    markup = '<table><thead><tr>' + ''.join('<th>' + html.escape(x) + '</th>' for x in headings) + '</tr></thead><tbody>'
    for row in table_rows:
        markup += '<tr>' + ''.join('<td>' + html.escape(str(x)) + '</td>' for x in row) + '</tr>'
    display(HTML(markup + '</tbody></table>'))
    print("Both workloads meet the prespecified criterion:", report["both_workloads_meet_prespecified_criterion"], flush=True)
    print("Full secondary results and calibration costs are in analysis.json. Raw-record audit is still required.", flush=True)
else:
    print("Analysis is not available yet. Exporting available checkpoints and diagnostics.", flush=True)

# Export is read-only with respect to the scientific run folder.
print("Collecting export file list...", flush=True)
all_paths = sorted(p for p in RUN_DIR.rglob("*") if p.is_file() and p.name != "RUNNING.lock")
inventory = []
paths = []
for path in all_paths:
    is_raw = path.name.endswith(".raw.jsonl.gz")
    included = bool(EXPORT_RAW or not is_raw)
    inventory.append(dict(path=str(path.relative_to(RUN_DIR)), bytes=path.stat().st_size,
                          included=included, raw_trace=is_raw))
    if included:
        paths.append(path)
byte_count = sum(item["bytes"] for item in inventory if item["included"])
raw_excluded = sum(item["raw_trace"] and not item["included"] for item in inventory)
flavor = "full" if EXPORT_RAW else "review"
archive_name = "P1_Qoala_Q12_" + RUN_DIR.name + "_" + flavor + ".zip"
ARCHIVE = PROJECT_DIR / archive_name
print(f"Exporting {len(paths):,} files ({byte_count / 1024**2:.1f} MiB before ZIP). "
      f"Raw trace files retained on server and omitted here: {raw_excluded:,}", flush=True)
temporary_archive = ARCHIVE.with_name(ARCHIVE.name + ".partial-" + str(os.getpid()))
started_export = time.monotonic()
last_export_print = started_export
with zipfile.ZipFile(temporary_archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:
    for index, path in enumerate(paths, 1):
        compression = zipfile.ZIP_STORED if path.suffix == ".gz" else zipfile.ZIP_DEFLATED
        archive.write(path, arcname=str(Path(RUN_DIR.name) / path.relative_to(RUN_DIR)), compress_type=compression)
        if index == 1 or index % 100 == 0 or time.monotonic() - last_export_print >= 10 or index == len(paths):
            print(f"Archived {index:,}/{len(paths):,}", flush=True)
            last_export_print = time.monotonic()
    archive.writestr("EXPORT_MANIFEST.json", json.dumps(dict(
        run_dir=str(RUN_DIR), archive_kind=flavor, source_bundle_sha256=SOURCE_BUNDLE_SHA256,
        raw_trace_files_excluded=raw_excluded,
        note="Review archives cannot independently verify omitted raw traces. Keep original raw files for audit.",
        files=inventory), indent=2))
os.replace(temporary_archive, ARCHIVE)
print(f"Results ZIP: {ARCHIVE} ({ARCHIVE.stat().st_size / 1024**2:.1f} MiB)", flush=True)
download = Path.cwd() / archive_name
if download.resolve() != ARCHIVE.resolve():
    print("Copying archive into the notebook working directory for download...", flush=True)
    shutil.copy2(ARCHIVE, download)
display(FileLink(archive_name))
print("Share the review ZIP first. Keep the original run directory for resume and full trace audit.", flush=True)


## After this run

- Share the review ZIP from the last cell, including negative or inconclusive outcomes.
- Keep the complete run folder. A review ZIP alone is insufficient for a complete raw-record audit or server resume.
- Reopening this notebook and running all cells with the default `RESUME_RUN_DIR="auto"` verifies and reuses the same run.
  You may change `WORKERS` when resuming. If the run already completed, no simulations are repeated.
- To create an independent new run deliberately, set `RESUME_RUN_DIR=None`. Do not delete protocol files or modify
  completed checkpoints to bypass a version mismatch.
- Q12 does not need Q6/Q8 result-folder paths. It needs the existing clean pinned Qoala checkout and the validated environment.
